In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Heart Disease Prediction with PCA and Best Model Selection
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv("./heart.csv")
print(df.head())

# Separate features and target
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# -----------------------------
# 2. Encode categorical variables
# -----------------------------
X_encoded = pd.get_dummies(X, drop_first=True)
print(X_encoded.head())

# -----------------------------
# 3. Remove outliers using Z-score
# -----------------------------
scaler_temp = StandardScaler()
z_scores = scaler_temp.fit_transform(X_encoded)
threshold = 3
mask = (np.abs(z_scores) < threshold).all(axis=1)

X_filtered = X_encoded[mask]
y_filtered = y[mask]

print("Filtered X shape:", X_filtered.shape)
print("Filtered y shape:", y_filtered.shape)

# -----------------------------
# 4. Scale features (again)
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# -----------------------------
# 5. Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_filtered, test_size=0.2, random_state=20
)

# -----------------------------
# 6. Find best model using GridSearchCV
# -----------------------------
model_params = {
    'SVM': {
        'model': SVC(),
        'params': {
            'C': list(range(1, 20)),
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'gamma': ['auto']
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(),
        'params': {
            'penalty': ['l1', 'l2'],
            'C': list(range(1, 20)),
            'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
            'multi_class': ['auto', 'ovr', 'multinomial']
        }
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': list(range(50, 151, 10)),
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    }
}

scores = []
best_models = {}

for model_name, model_info in model_params.items():
    clf = GridSearchCV(model_info['model'], model_info['params'], cv=5, return_train_score=False)
    clf.fit(X_train, y_train)
    scores.append({
        "model": model_name,
        "best_score": clf.best_score_,
        "best_params": clf.best_params_
    })
    best_models[model_name] = clf.best_estimator_

df_scores = pd.DataFrame(scores)
print(df_scores)

# -----------------------------
# 7. Evaluate best model (before PCA)
# -----------------------------
# Pick the model with the highest cross-validation score
best_model_name = df_scores.loc[df_scores['best_score'].idxmax(), 'model']
best_model = best_models[best_model_name]

print(f"Best model before PCA: {best_model_name}")
print("Test accuracy before PCA:", best_model.score(X_test, y_test))

# -----------------------------
# 8. Apply PCA (retain 95% variance)
# -----------------------------
pca = PCA(0.95)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)  # Transform test data only

# Retrain best model on PCA-reduced data
best_model.fit(X_train_pca, y_train)
pca_accuracy = best_model.score(X_test_pca, y_test)

print("Test accuracy after PCA:", pca_accuracy)
print("Number of PCA components:", pca.n_components_)
